In [43]:
import google.generativeai as genai
import os

In [44]:
from google.cloud import aiplatform

# Google (Gemini)
aiplatform.init(project="LLM-XAI")

import google.generativeai as genai
import os

# Set your API key
api_key = "YOur APIgenai.configure(api_key=api_key)

# Initialize the model
model = genai.GenerativeModel('gemini-3.1-flash-lite')

# Test the connection
response = model.generate_content("Hello, Gemini!")
print(response.text)

Hello! How can I help you today?


In [45]:
from dataclasses import dataclass
from typing import List, Dict
from enum import Enum

class TestType(Enum):
    """Types of explainability tests"""
    FACT_RETRIEVAL = "fact_retrieval"
    NEGATION = "negation"
    IRRELEVANT_INFO = "irrelevant_info"
    FEATURE_INTERACTION = "feature_interaction"
    REASONING_CHAIN = "reasoning_chain"
    INSTRUCTION_FOLLOWING = "instruction_following"

@dataclass
class GeminiTestCase:
    """Test case for Gemini explainability"""
    name: str
    prompt: str
    expected_output: str
    important_tokens: List[str]  # Ground truth - tokens that should matter
    unimportant_tokens: List[str]  # Tokens that shouldn't matter
    test_type: TestType
    description: str

# Create comprehensive test suite
gemini_test_cases = [
    # Test 1: Simple Fact Retrieval
    GeminiTestCase(
        name="Simple Fact Retrieval",
        prompt="The capital of France is",
        expected_output="Paris",
        important_tokens=["capital", "France"],
        unimportant_tokens=["The", "of", "is"],
        test_type=TestType.FACT_RETRIEVAL,
        description="Model should identify 'France' as the key token for retrieving its capital"
    ),
    
    # Test 2: Negation Handling
    GeminiTestCase(
        name="Negation Reversal",
        prompt="The movie was NOT good. What was the sentiment?",
        expected_output="negative",
        important_tokens=["NOT", "good"],
        unimportant_tokens=["The", "movie", "was", "What", "sentiment"],
        test_type=TestType.NEGATION,
        description="Model should identify 'NOT' as critical for reversing sentiment"
    ),
    
    # Test 3: Irrelevant Information
    GeminiTestCase(
        name="Filtering Irrelevant Info",
        prompt="The capital of France is Paris. The weather today is sunny. What is the capital of France?",
        expected_output="Paris",
        important_tokens=["capital", "France"],
        unimportant_tokens=["weather", "sunny", "today"],
        test_type=TestType.IRRELEVANT_INFO,
        description="Model should ignore weather information and focus on the question"
    ),
    
    # Test 4: Feature Interaction
    GeminiTestCase(
        name="Adjective-Noun Interaction",
        prompt="The old man walked slowly. Who walked slowly?",
        expected_output="the old man",
        important_tokens=["old", "man", "walked", "slowly"],
        unimportant_tokens=["The", "Who"],
        test_type=TestType.FEATURE_INTERACTION,
        description="Model should understand that 'old' modifies 'man'"
    ),
    
    # Test 5: Reasoning Chain
    GeminiTestCase(
        name="Multi-Step Reasoning",
        prompt="If all dogs are animals, and Fido is a dog, then Fido is an animal. Is Fido an animal?",
        expected_output="yes",
        important_tokens=["dogs", "animals", "Fido", "dog"],
        unimportant_tokens=["If", "all", "and", "then", "is", "an"],
        test_type=TestType.REASONING_CHAIN,
        description="Model should follow logical chain to reach conclusion"
    ),
    
    # Test 6: Instruction Following
    GeminiTestCase(
        name="Instruction Following",
        prompt="Respond with only one word. What is the capital of France?",
        expected_output="Paris",
        important_tokens=["one", "word", "capital", "France"],
        unimportant_tokens=["Respond", "with", "only", "What", "is"],
        test_type=TestType.INSTRUCTION_FOLLOWING,
        description="Model should identify instruction constraint and follow it"
    ),
]

print(f"Created {len(gemini_test_cases)} test cases")
for i, test in enumerate(gemini_test_cases):
    print(f"{i+1}. {test.name} ({test.test_type.value})")

Created 6 test cases
1. Simple Fact Retrieval (fact_retrieval)
2. Negation Reversal (negation)
3. Filtering Irrelevant Info (irrelevant_info)
4. Adjective-Noun Interaction (feature_interaction)
5. Multi-Step Reasoning (reasoning_chain)
6. Instruction Following (instruction_following)


In [46]:
class GeminiTestDataset:
    """Dataset of test cases for Gemini"""
    
    def __init__(self, test_cases: List[GeminiTestCase]):
        self.test_cases = test_cases
        self.results = []
    
    def __len__(self):
        return len(self.test_cases)
    
    def __getitem__(self, idx):
        return self.test_cases[idx]
    
    def get_by_type(self, test_type: TestType) -> List[GeminiTestCase]:
        """Get all test cases of a specific type"""
        return [tc for tc in self.test_cases if tc.test_type == test_type]
    
    def tokenize(self, text: str) -> List[str]:
        """Simple tokenization (split by spaces)"""
        return text.split()

# Create dataset
dataset = GeminiTestDataset(gemini_test_cases)
print(f"Dataset created with {len(dataset)} test cases")

Dataset created with 6 test cases


In [68]:
dataset.__getitem__(1)  # Get first test case

GeminiTestCase(name='Negation Reversal', prompt='The movie was NOT good. What was the sentiment?', expected_output='negative', important_tokens=['NOT', 'good'], unimportant_tokens=['The', 'movie', 'was', 'What', 'sentiment'], test_type=<TestType.NEGATION: 'negation'>, description="Model should identify 'NOT' as critical for reversing sentiment")

In [47]:
from abc import ABC, abstractmethod
from typing import Dict, Tuple
import time

class GeminiAttributionMethod(ABC):
    """Base class for Gemini attribution methods"""
    
    def __init__(self, model_name: str = "gemini-3.5-flash"):
        self.model = genai.GenerativeModel(model_name)
        self.model_name = model_name
        self.attribution_time = 0
    
    @abstractmethod
    def attribute(self, prompt: str) -> Dict[str, float]:
        """
        Compute attribution scores for each token in the prompt.
        
        Args:
            prompt: Input text
            
        Returns:
            Dictionary mapping token -> importance score (0-1)
        """
        pass
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization"""
        return text.split()
    
    def _get_response(self, prompt: str) -> str:
        """Get Gemini response"""
        try:
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Error getting response: {e}")
            return ""

In [69]:
class PerturbationAttribution(GeminiAttributionMethod):
    """
    Perturbation-based attribution for Gemini.
    
    Idea: Remove each token and measure how much the output changes.
    Similar to LIME for LLMs.
    """
    
    def attribute(self, prompt: str) -> Dict[str, float]:
        """
        For each token, remove it and measure impact on output.
        """
        start_time = time.time()
        tokens = self._tokenize(prompt)
        attributions = {}
        
        # Get baseline response
        baseline_response = self._get_response(prompt)
        baseline_length = len(baseline_response.split())
        
        print(f"Baseline response: {baseline_response[:100]}...")
        print(f"Baseline length: {baseline_length} tokens")
        
        # For each token, remove it and measure change
        for i, token in enumerate(tokens):
            # Create prompt without this token
            tokens_without = tokens[:i] + tokens[i+1:]
            prompt_without = " ".join(tokens_without)
            
            # Get response without this token
            response_without = self._get_response(prompt_without)
            response_without_length = len(response_without.split())
            
            # Attribution = how much output changed
            # Measure: change in response length
            length_change = abs(baseline_length - response_without_length)
            
            # Normalize to 0-1
            attribution = min(length_change / baseline_length, 1.0)
            attributions[token] = attribution
            
            print(f"  Token '{token}': attribution = {attribution:.4f}")
        
        self.attribution_time = time.time() - start_time
        return attributions

# Example usage
perturbation_method = PerturbationAttribution()
test_prompt = "The capital of Pakistan is"
attributions = perturbation_method.attribute(test_prompt)

print("\nAttributions:")
for token, score in sorted(attributions.items(), key=lambda x: x[1], reverse=True):
    print(f"  {token}: {score:.4f}")

Baseline response: The capital of Pakistan is **Islamabad**....
Baseline length: 6 tokens
  Token 'The': attribution = 0.0000
  Token 'capital': attribution = 1.0000
  Token 'of': attribution = 0.0000
  Token 'Pakistan': attribution = 1.0000
  Token 'is': attribution = 0.0000

Attributions:
  capital: 1.0000
  Pakistan: 1.0000
  The: 0.0000
  of: 0.0000
  is: 0.0000


In [49]:
class TokenMaskingAttribution(GeminiAttributionMethod):
    """
    Token masking attribution for Gemini.
    
    Idea: Replace each token with [MASK] and measure output change.
    """
    
    def attribute(self, prompt: str) -> Dict[str, float]:
        """
        For each token, replace with [MASK] and measure impact.
        """
        start_time = time.time()
        tokens = self._tokenize(prompt)
        attributions = {}
        
        # Get baseline response
        baseline_response = self._get_response(prompt)
        
        print(f"Baseline response: {baseline_response[:100]}...")
        
        # For each token, mask it
        for i, token in enumerate(tokens):
            # Create prompt with masked token
            tokens_masked = tokens.copy()
            tokens_masked[i] = "[MASK]"
            prompt_masked = " ".join(tokens_masked)
            
            # Get response with masked token
            response_masked = self._get_response(prompt_masked)
            
            # Attribution = semantic similarity between responses
            # Simple measure: exact match (0 or 1)
            if response_masked == baseline_response:
                attribution = 0.0  # Token doesn't matter
            else:
                attribution = 1.0  # Token matters
            
            attributions[token] = attribution
            
            print(f"  Token '{token}': attribution = {attribution:.4f}")
        
        self.attribution_time = time.time() - start_time
        return attributions

In [71]:
class GradientApproximationAttribution(GeminiAttributionMethod):
    """
    Gradient-based attribution approximation for Gemini.
    
    Idea: Approximate gradients by measuring sensitivity to small changes.
    """
    
    def attribute(self, prompt: str) -> Dict[str, float]:
        """
        Approximate gradients by measuring output sensitivity.
        """
        start_time = time.time()
        tokens = self._tokenize(prompt)
        attributions = {}
        
        # Get baseline response
        baseline_response = self._get_response(prompt)
        
        print(f"Baseline response: {baseline_response[:100]}...")
        
        # For each token, measure sensitivity
        for i, token in enumerate(tokens):
            # Create slightly perturbed prompts
            perturbations = [
                self._perturb_token(tokens, i, "remove"),
                self._perturb_token(tokens, i, "duplicate"),
                self._perturb_token(tokens, i, "replace"),
            ]
            
            # Measure how much responses differ
            differences = []
            for perturbed_prompt in perturbations:
                response = self._get_response(perturbed_prompt)
                diff = self._compute_difference(baseline_response, response)
                differences.append(diff)
            
            # Attribution = average difference
            attribution = sum(differences) / len(differences)
            attributions[token] = attribution
            
            print(f"  Token '{token}': attribution = {attribution:.4f}")
        
        self.attribution_time = time.time() - start_time
        return attributions
    
    def _perturb_token(self, tokens: List[str], idx: int, method: str) -> str:
        """Perturb a token in different ways"""
        tokens_copy = tokens.copy()
        
        if method == "remove":
            tokens_copy = tokens_copy[:idx] + tokens_copy[idx+1:]
        elif method == "duplicate":
            tokens_copy = tokens_copy[:idx+1] + tokens_copy[idx:]
        elif method == "replace":
            tokens_copy[idx] = "[UNKNOWN]"
        
        return " ".join(tokens_copy)
    
    def _compute_difference(self, response1: str, response2: str) -> float:
        """Compute difference between two responses"""
        # Simple: Levenshtein distance normalized
        from difflib import SequenceMatcher
        ratio = SequenceMatcher(None, response1, response2).ratio()
        return 1.0 - ratio  # 0 = same, 1 = completely different
    


In [51]:
from typing import Dict, List
import numpy as np
from dataclasses import dataclass

@dataclass
class MetricResult:
    """Result of a single metric evaluation"""
    metric_name: str
    score: float
    details: Dict = None

class GeminiEvaluationMetric:
    """Base class for evaluation metrics"""
    
    def evaluate(self, 
                 attributions: Dict[str, float],
                 important_tokens: List[str],
                 unimportant_tokens: List[str]) -> MetricResult:
        """
        Evaluate how well attributions match ground truth.
        
        Args:
            attributions: Dict mapping token -> importance score
            important_tokens: Ground truth important tokens
            unimportant_tokens: Ground truth unimportant tokens
            
        Returns:
            MetricResult with score and details
        """
        raise NotImplementedError

class PrecisionAtKMetric(GeminiEvaluationMetric):
    """
    Precision@K: What fraction of top-K tokens are important?
    
    Formula: (# of important tokens in top-K) / K
    
    Example:
    - Top-3 tokens: ["France", "capital", "The"]
    - Important tokens: ["France", "capital"]
    - Precision@3 = 2/3 = 0.667
    """
    
    def __init__(self, k: int = 3):
        self.k = k
    
    def evaluate(self, 
                 attributions: Dict[str, float],
                 important_tokens: List[str],
                 unimportant_tokens: List[str]) -> MetricResult:
        """
        Get top-K tokens by attribution score.
        Check how many are in important_tokens.
        """
        # Sort tokens by attribution score (descending)
        sorted_tokens = sorted(attributions.items(), 
                              key=lambda x: x[1], 
                              reverse=True)
        
        # Get top-K tokens
        top_k_tokens = [token for token, score in sorted_tokens[:self.k]]
        
        # Count how many are important
        correct = sum(1 for token in top_k_tokens if token in important_tokens)
        
        # Precision@K
        precision = correct / self.k if self.k > 0 else 0.0
        
        details = {
            "top_k_tokens": top_k_tokens,
            "correct_count": correct,
            "k": self.k
        }
        
        return MetricResult(
            metric_name=f"Precision@{self.k}",
            score=precision,
            details=details
        )

class RecallAtKMetric(GeminiEvaluationMetric):
    """
    Recall@K: What fraction of important tokens are in top-K?
    
    Formula: (# of important tokens in top-K) / (total important tokens)
    
    Example:
    - Top-3 tokens: ["France", "capital", "The"]
    - Important tokens: ["France", "capital"]
    - Recall@3 = 2/2 = 1.0 (all important tokens found)
    """
    
    def __init__(self, k: int = 3):
        self.k = k
    
    def evaluate(self, 
                 attributions: Dict[str, float],
                 important_tokens: List[str],
                 unimportant_tokens: List[str]) -> MetricResult:
        """
        Get top-K tokens by attribution score.
        Check what fraction of important tokens are included.
        """
        # Sort tokens by attribution score (descending)
        sorted_tokens = sorted(attributions.items(), 
                              key=lambda x: x[1], 
                              reverse=True)
        
        # Get top-K tokens
        top_k_tokens = [token for token, score in sorted_tokens[:self.k]]
        
        # Count how many important tokens are in top-K
        found = sum(1 for token in important_tokens if token in top_k_tokens)
        
        # Recall@K
        total_important = len(important_tokens)
        recall = found / total_important if total_important > 0 else 0.0
        
        details = {
            "top_k_tokens": top_k_tokens,
            "found_count": found,
            "total_important": total_important,
            "k": self.k
        }
        
        return MetricResult(
            metric_name=f"Recall@{self.k}",
            score=recall,
            details=details
        )

class F1ScoreMetric(GeminiEvaluationMetric):
    """
    F1 Score: Harmonic mean of Precision and Recall
    
    Formula: 2 * (Precision * Recall) / (Precision + Recall)
    
    Balances precision and recall.
    """
    
    def __init__(self, k: int = 3):
        self.k = k
        self.precision_metric = PrecisionAtKMetric(k)
        self.recall_metric = RecallAtKMetric(k)
    
    def evaluate(self, 
                 attributions: Dict[str, float],
                 important_tokens: List[str],
                 unimportant_tokens: List[str]) -> MetricResult:
        """
        Compute F1 score from precision and recall.
        """
        precision_result = self.precision_metric.evaluate(
            attributions, important_tokens, unimportant_tokens
        )
        recall_result = self.recall_metric.evaluate(
            attributions, important_tokens, unimportant_tokens
        )
        
        precision = precision_result.score
        recall = recall_result.score
        
        # F1 = 2 * (P * R) / (P + R)
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0
        
        details = {
            "precision": precision,
            "recall": recall,
            "k": self.k
        }
        
        return MetricResult(
            metric_name=f"F1@{self.k}",
            score=f1,
            details=details
        )

class TokenClassificationMetric(GeminiEvaluationMetric):
    """
    Token Classification: Classify each token as important or unimportant.
    
    Uses a threshold to classify tokens.
    Computes accuracy, precision, recall, F1 for binary classification.
    """
    
    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
    
    def evaluate(self, 
                 attributions: Dict[str, float],
                 important_tokens: List[str],
                 unimportant_tokens: List[str]) -> MetricResult:
        """
        Classify tokens as important if attribution > threshold.
        Compute metrics against ground truth.
        """
        # Create predictions and ground truth
        predictions = []
        ground_truth = []
        
        for token, score in attributions.items():
            # Predict: 1 if score > threshold, 0 otherwise
            pred = 1 if score > self.threshold else 0
            predictions.append(pred)
            
            # Ground truth: 1 if important, 0 if unimportant
            if token in important_tokens:
                gt = 1
            elif token in unimportant_tokens:
                gt = 0
            else:
                # Token not in either list - skip
                continue
            
            ground_truth.append(gt)
        
        # Compute metrics
        predictions = np.array(predictions[:len(ground_truth)])
        ground_truth = np.array(ground_truth)
        
        # Accuracy
        accuracy = np.mean(predictions == ground_truth)
        
        # Precision, Recall, F1
        tp = np.sum((predictions == 1) & (ground_truth == 1))
        fp = np.sum((predictions == 1) & (ground_truth == 0))
        fn = np.sum((predictions == 0) & (ground_truth == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        details = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "threshold": self.threshold,
            "true_positives": int(tp),
            "false_positives": int(fp),
            "false_negatives": int(fn)
        }
        
        return MetricResult(
            metric_name="Token Classification",
            score=f1,  # Use F1 as main score
            details=details
        )

class RankCorrelationMetric(GeminiEvaluationMetric):
    """
    Rank Correlation: How well do predicted rankings match ground truth?
    
    Uses Spearman's rank correlation coefficient.
    """
    
    def evaluate(self, 
                 attributions: Dict[str, float],
                 important_tokens: List[str],
                 unimportant_tokens: List[str]) -> MetricResult:
        """
        Compute rank correlation between predicted and ground truth rankings.
        """
        from scipy.stats import spearmanr
        
        # Create ground truth scores
        ground_truth_scores = {}
        for token in attributions.keys():
            if token in important_tokens:
                ground_truth_scores[token] = 1.0
            elif token in unimportant_tokens:
                ground_truth_scores[token] = 0.0
            else:
                ground_truth_scores[token] = 0.5  # Unknown
        
        # Get predicted scores
        predicted_scores = list(attributions.values())
        true_scores = [ground_truth_scores[token] for token in attributions.keys()]
        
        # Compute Spearman correlation
        correlation, p_value = spearmanr(predicted_scores, true_scores)
        
        details = {
            "correlation": correlation,
            "p_value": p_value
        }
        
        return MetricResult(
            metric_name="Rank Correlation",
            score=correlation,
            details=details
        )

In [52]:
from dataclasses import dataclass, field
from typing import Dict, List
import pandas as pd
from datetime import datetime

@dataclass
class EvaluationResult:
    """Result of evaluating an attribution method on a test case"""
    test_name: str
    test_type: str
    method_name: str
    metrics: Dict[str, float]  # metric_name -> score
    metric_details: Dict[str, Dict] = field(default_factory=dict)
    attributions: Dict[str, float] = field(default_factory=dict)
    attribution_time: float = 0.0
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for DataFrame"""
        result_dict = {
            "test_name": self.test_name,
            "test_type": self.test_type,
            "method_name": self.method_name,
            "attribution_time": self.attribution_time,
            "timestamp": self.timestamp,
        }
        # Add all metrics
        for metric_name, score in self.metrics.items():
            result_dict[metric_name] = score
        
        return result_dict

In [62]:
class GeminiEvaluationPipeline:
    """
    Pipeline for evaluating Gemini explainability.
    
    Workflow:
    1. For each test case
    2. For each attribution method
    3. Compute attributions
    4. Evaluate with all metrics
    5. Store results
    """
    
    def __init__(self, 
                 dataset: GeminiTestDataset,
                 attribution_methods: List[GeminiAttributionMethod],
                 evaluation_metrics: List[GeminiEvaluationMetric],
                 name: str = "gemini_evaluation"):
        """
        Initialize pipeline.
        
        Args:
            dataset: GeminiTestDataset with test cases
            attribution_methods: List of attribution methods to test
            evaluation_metrics: List of metrics to compute
            name: Name for this evaluation run
        """
        self.dataset = dataset
        self.attribution_methods = attribution_methods
        self.evaluation_metrics = evaluation_metrics
        self.name = name
        self.results = []
        self.results_df = None
    
    def run(self, verbose: bool = True) -> pd.DataFrame:
        """
        Run the evaluation pipeline.
        
        Args:
            verbose: Print progress information
            
        Returns:
            DataFrame with all results
        """
        print(f"\n{'='*80}")
        print(f"GEMINI EXPLAINABILITY EVALUATION PIPELINE")
        print(f"{'='*80}")
        print(f"Name: {self.name}")
        print(f"Test cases: {len(self.dataset)}")
        print(f"Attribution methods: {len(self.attribution_methods)}")
        print(f"Evaluation metrics: {len(self.evaluation_metrics)}")
        print(f"Total evaluations: {len(self.dataset) * len(self.attribution_methods)}")
        print(f"{'='*80}\n")
        
        # Iterate through all combinations
        total_evals = len(self.dataset) * len(self.attribution_methods)
        current_eval = 0
        
        for test_case in self.dataset:
            for method in self.attribution_methods:
                current_eval += 1
                
                if verbose:
                    print(f"\n[{current_eval}/{total_evals}] Evaluating {method.__class__.__name__} on {test_case.name}")

                
                
                # Run evaluation
                result = self._evaluate_single(test_case, method, verbose)
                self.results.append(result)
        
        # Convert to DataFrame
        self.results_df = pd.DataFrame([r.to_dict() for r in self.results])
        
        if verbose:
            print(f"\n{'='*80}")
            print(f"EVALUATION COMPLETE")
            print(f"Total results: {len(self.results)}")
            print(f"{'='*80}\n")
        
        return self.results_df
    
    def _evaluate_single(self, 
                    test_case: GeminiTestCase,
                    method: GeminiAttributionMethod,
                    verbose: bool = True) -> EvaluationResult:
        """ 
        Evaluate a single attribution method on a single test case.
        
        Workflow:
        1. Get attributions from the method
        2. Evaluate attributions against ground truth using all metrics
        3. Package results
        """
        
        # ============================================================
        # STEP 1: GET ATTRIBUTIONS
        # ============================================================
        if verbose:
            print(f"  Getting attributions...")
        delay = 60  # seconds
        time.sleep(delay)  # To avoid rate limits
        attributions = method.attribute(test_case.prompt)
        attribution_time = method.attribution_time
        
        if verbose:
            print(f"  Attribution time: {attribution_time:.2f}s")
            top_3 = sorted(attributions.items(), key=lambda x: x[1], reverse=True)[:3]
            print(f"  Top-3 tokens: {top_3}")
        
        # ============================================================
        # STEP 2: EVALUATE WITH ALL METRICS
        # ============================================================
        if verbose:
            print(f"  Computing {len(self.evaluation_metrics)} metrics...")
        
        metrics = {}
        metric_details = {}
        
        for metric in self.evaluation_metrics:
            # Evaluate this metric
            metric_result = metric.evaluate(
                attributions=attributions,
                important_tokens=test_case.important_tokens,
                unimportant_tokens=test_case.unimportant_tokens
            )
            
            # Store score and details
            metrics[metric_result.metric_name] = metric_result.score
            metric_details[metric_result.metric_name] = metric_result.details
            
            if verbose:
                print(f"    {metric_result.metric_name}: {metric_result.score:.4f}")
        
        # ============================================================
        # STEP 3: PACKAGE RESULTS
        # ============================================================
        result = EvaluationResult(
            test_name=test_case.name,
            test_type=test_case.test_type.value,
            method_name=method.__class__.__name__,
            metrics=metrics,
            metric_details=metric_details,
            attributions=attributions,
            attribution_time=attribution_time
        )
        
        return result

In [63]:
# ============================================================
# SETUP: Initialize everything
# ============================================================

# 1. Create dataset (you already have this)
dataset = GeminiTestDataset(gemini_test_cases)

# 2. Create attribution methods
attribution_methods = [
    PerturbationAttribution(model_name="gemini-3.1-flash-lite"),
    TokenMaskingAttribution(model_name="gemini-3.1-flash-lite"),
    GradientApproximationAttribution(model_name="gemini-3.1-flash-lite"),
]

# 3. Create evaluation metrics
evaluation_metrics = [
    PrecisionAtKMetric(k=3),
    RecallAtKMetric(k=3),
    F1ScoreMetric(k=3),
    TokenClassificationMetric(threshold=0.5),
    RankCorrelationMetric(),
]

# 4. Create pipeline
pipeline = GeminiEvaluationPipeline(
    dataset=dataset,
    attribution_methods=attribution_methods,
    evaluation_metrics=evaluation_metrics,
    name="gemini_xai_evaluation_v1"
)

# ============================================================
# RUN: Execute the full evaluation
# ============================================================

results_df = pipeline.run(verbose=True)

# ============================================================
# ANALYZE: Examine results
# ============================================================

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

# Show average scores by method
print("\nAverage Scores by Attribution Method:")
method_summary = results_df.groupby("method_name")[
    ["Precision@3", "Recall@3", "F1@3"]
].mean()
print(method_summary)

# Show average scores by test type
print("\nAverage Scores by Test Type:")
test_summary = results_df.groupby("test_type")[
    ["Precision@3", "Recall@3", "F1@3"]
].mean()
print(test_summary)

# Show best performing combinations
print("\nTop 5 Best Performing Combinations:")
top_results = results_df.nlargest(5, "F1@3")[
    ["test_name", "method_name", "F1@3", "attribution_time"]
]
print(top_results)

# Show worst performing combinations
print("\nBottom 5 Worst Performing Combinations:")
bottom_results = results_df.nsmallest(5, "F1@3")[
    ["test_name", "method_name", "F1@3", "attribution_time"]
]
print(bottom_results)


GEMINI EXPLAINABILITY EVALUATION PIPELINE
Name: gemini_xai_evaluation_v1
Test cases: 6
Attribution methods: 3
Evaluation metrics: 5
Total evaluations: 18


[1/18] Evaluating PerturbationAttribution on Simple Fact Retrieval
  Getting attributions...
Baseline response: The capital of France is Paris....
Baseline length: 6 tokens
  Token 'The': attribution = 0.0000
  Token 'capital': attribution = 1.0000
  Token 'of': attribution = 0.0000
  Token 'France': attribution = 1.0000
  Token 'is': attribution = 0.0000
  Attribution time: 4.12s
  Top-3 tokens: [('capital', 1.0), ('France', 1.0), ('The', 0.0)]
  Computing 5 metrics...
    Precision@3: 0.6667
    Recall@3: 1.0000
    F1@3: 0.8000
    Token Classification: 1.0000
    Rank Correlation: 1.0000

[2/18] Evaluating TokenMaskingAttribution on Simple Fact Retrieval
  Getting attributions...
Baseline response: The capital of France is Paris....
  Token 'The': attribution = 1.0000
  Token 'capital': attribution = 1.0000
  Token 'of': attrib

C:\Users\imran\AppData\Local\Temp\ipykernel_18512\3883734388.py:284: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  correlation, p_value = spearmanr(predicted_scores, true_scores)


Baseline response: Paris...
  Token 'The': attribution = 0.7500
  Token 'capital': attribution = 0.9013
  Token 'of': attribution = 0.7576
  Token 'France': attribution = 0.8791
  Token 'is': attribution = 0.7500
  Attribution time: 9.89s
  Top-3 tokens: [('capital', 0.9013320088233262), ('France', 0.879059829059829), ('of', 0.7575757575757575)]
  Computing 5 metrics...
    Precision@3: 0.6667
    Recall@3: 1.0000
    F1@3: 0.8000
    Token Classification: 0.5714
    Rank Correlation: 0.8885

[4/18] Evaluating PerturbationAttribution on Negation Reversal
  Getting attributions...
Baseline response: The sentiment is **negative**....
Baseline length: 4 tokens
  Token 'The': attribution = 0.0000
  Token 'movie': attribution = 1.0000
  Token 'was': attribution = 0.0000
  Token 'NOT': attribution = 0.0000
  Token 'good.': attribution = 1.0000
  Token 'What': attribution = 0.0000
  Token 'was': attribution = 0.0000
  Token 'the': attribution = 0.0000
  Token 'sentiment?': attribution = 1.000

In [64]:
print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)

# Show average scores by method
print("\nAverage Scores by Attribution Method:")
method_summary = results_df.groupby("method_name")[
    ["Precision@3", "Recall@3", "F1@3"]
].mean()
print(method_summary)

# Show average scores by test type
print("\nAverage Scores by Test Type:")
test_summary = results_df.groupby("test_type")[
    ["Precision@3", "Recall@3", "F1@3"]
].mean()
print(test_summary)

# Show best performing combinations
print("\nTop 5 Best Performing Combinations:")
top_results = results_df.nlargest(5, "F1@3")[
    ["test_name", "method_name", "F1@3", "attribution_time"]
]
print(top_results)

# Show worst performing combinations
print("\nBottom 5 Worst Performing Combinations:")
bottom_results = results_df.nsmallest(5, "F1@3")[
    ["test_name", "method_name", "F1@3", "attribution_time"]
]
print(bottom_results)


RESULTS SUMMARY

Average Scores by Attribution Method:
                                  Precision@3  Recall@3      F1@3
method_name                                                      
GradientApproximationAttribution     0.333333  0.416667  0.361905
PerturbationAttribution              0.277778  0.291667  0.276190
TokenMaskingAttribution              0.333333  0.375000  0.342857

Average Scores by Test Type:
                       Precision@3  Recall@3      F1@3
test_type                                             
fact_retrieval            0.555556  0.833333  0.666667
feature_interaction       0.555556  0.416667  0.476190
instruction_following     0.000000  0.000000  0.000000
irrelevant_info           0.222222  0.333333  0.266667
negation                  0.222222  0.333333  0.266667
reasoning_chain           0.333333  0.250000  0.285714

Top 5 Best Performing Combinations:
                     test_name                       method_name      F1@3  \
0        Simple Fact Retrieva